### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

In [11]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
menu = db["menu"]

# Jalankan ulang dengan aman: kosongkan koleksi agar tidak terjadi duplikasi
menu.delete_many({})
dokumen_menu = [
    {"nama_menu":"Espresso","harga":18000,"kategori":"Kopi","tingkat_kepahitan":5,"stok":20},
    {"nama_menu":"Americano","harga":22000,"kategori":"Kopi","tingkat_kepahitan":4,"stok":25},
    {"nama_menu":"Cappuccino","harga":28000,"kategori":"Kopi","tingkat_kepahitan":3,"stok":18},
    {"nama_menu":"Cafe Latte","harga":27000,"kategori":"Kopi","tingkat_kepahitan":2,"stok":22},
    {"nama_menu":"Kopi Tubruk","harga":15000,"kategori":"Kopi","tingkat_kepahitan":4,"stok":30},
    {"nama_menu":"Mocha","harga":30000,"kategori":"Kopi","tingkat_kepahitan":3,"stok":15},
    {"nama_menu":"Nasi Goreng Nusantara","harga":28000,"kategori":"Makanan","pedas_level":2,"stok":12},
    {"nama_menu":"Mie Goreng Jawa","harga":25000,"kategori":"Makanan","pedas_level":3,"stok":10},
    {"nama_menu":"Roti Bakar Cokelat","harga":18000,"kategori":"Makanan","pedas_level":0,"stok":16},
    {"nama_menu":"Pisang Goreng","harga":15000,"kategori":"Makanan","pedas_level":0,"stok":20},
    {"nama_menu":"Chicken Rice Bowl","harga":30000,"kategori":"Makanan","pedas_level":1,"stok":9},
    {"nama_menu":"Singkong Keju","harga":20000,"kategori":"Makanan","pedas_level":0,"stok":14}
]
hasil_insert = menu.insert_many(dokumen_menu)
print(f"Berhasil menyisipkan {len(hasil_insert.inserted_ids)} dokumen")
print("Jumlah dokumen:", menu.count_documents({}))

Berhasil menyisipkan 12 dokumen
Jumlah dokumen: 12


**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

In [12]:
# 1. Kopi dengan tingkat kepahitan di atas 3
for item in menu.find({"kategori":"Kopi", "tingkat_kepahitan":{"$gt":3}}):
    print(item)

# 2. Harga antara 15000 dan 30000 (inklusif)
for item in menu.find({"harga":{"$gte":15000, "$lte":30000}}):
    print(item)

# 3. Dokumen yang tidak memiliki field pedas_level
for item in menu.find({"pedas_level":{"$exists":False}}):
    print(item)

{'_id': ObjectId('6aeb380c409d172c973fa8f9'), 'nama_menu': 'Espresso', 'harga': 18000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 20}
{'_id': ObjectId('6aeb380c409d172c973fa8fa'), 'nama_menu': 'Americano', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 25}
{'_id': ObjectId('6aeb380c409d172c973fa8fd'), 'nama_menu': 'Kopi Tubruk', 'harga': 15000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 30}
{'_id': ObjectId('6aeb380c409d172c973fa8f9'), 'nama_menu': 'Espresso', 'harga': 18000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 20}
{'_id': ObjectId('6aeb380c409d172c973fa8fa'), 'nama_menu': 'Americano', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 25}
{'_id': ObjectId('6aeb380c409d172c973fa8fb'), 'nama_menu': 'Cappuccino', 'harga': 28000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 18}
{'_id': ObjectId('6aeb380c409d172c973fa8fc'), 'nama_menu': 'Cafe Latte', 'harga': 27000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 's

**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

In [13]:
# Tambahkan status Tersedia pada seluruh dokumen
hasil_update = menu.update_many({}, {"$set":{"status":"Tersedia"}})
print("Dokumen dimodifikasi:", hasil_update.modified_count)

# Ubah status dua menu pilihan menjadi Habis
menu.update_one({"nama_menu":"Espresso"}, {"$set":{"status":"Habis"}})
menu.update_one({"nama_menu":"Nasi Goreng Nusantara"}, {"$set":{"status":"Habis"}})

for item in menu.find({}, {"_id":0, "nama_menu":1, "status":1}):
    print(item)

Dokumen dimodifikasi: 12
{'nama_menu': 'Espresso', 'status': 'Habis'}
{'nama_menu': 'Americano', 'status': 'Tersedia'}
{'nama_menu': 'Cappuccino', 'status': 'Tersedia'}
{'nama_menu': 'Cafe Latte', 'status': 'Tersedia'}
{'nama_menu': 'Kopi Tubruk', 'status': 'Tersedia'}
{'nama_menu': 'Mocha', 'status': 'Tersedia'}
{'nama_menu': 'Nasi Goreng Nusantara', 'status': 'Habis'}
{'nama_menu': 'Mie Goreng Jawa', 'status': 'Tersedia'}
{'nama_menu': 'Roti Bakar Cokelat', 'status': 'Tersedia'}
{'nama_menu': 'Pisang Goreng', 'status': 'Tersedia'}
{'nama_menu': 'Chicken Rice Bowl', 'status': 'Tersedia'}
{'nama_menu': 'Singkong Keju', 'status': 'Tersedia'}


**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

In [14]:
# Membaca seluruh dokumen MongoDB dan membentuk Spark DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder.appName("KatalogKopiNusantara").getOrCreate()
data_menu = list(menu.find({}, {"_id":0}))

# Field kategori-spesifik dipertahankan; field yang tidak ada akan menjadi null
# saat Spark menyusun skema dari dokumen-dokumen tersebut.
df_menu = spark.createDataFrame(data_menu)
df_menu.printSchema()

print("Rata-rata harga per kategori:")
df_menu.groupBy("kategori").agg(avg(col("harga")).alias("rata_rata_harga")).show()

26/11/04 20:50:07 WARN Utils: Your hostname, LAPTOP-UGSS09T8 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/04 20:50:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 20:50:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- status: string (nullable = true)
 |-- stok: long (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- pedas_level: long (nullable = true)

Rata-rata harga per kategori:


+--------+------------------+
|kategori|   rata_rata_harga|
+--------+------------------+
|    Kopi|23333.333333333332|
| Makanan|22666.666666666668|
+--------+------------------+



**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

Dalam proses mengonversi data MongoDB ke Spark DataFrame, tantangan yang ditemui adalah struktur dokumen MongoDB tidak selalu sama. Pada katalog ini, menu kategori Kopi memiliki field tingkat_kepahitan, sedangkan menu kategori Makanan memiliki field pedas_level. Akibatnya, ketika data digabungkan menjadi bentuk tabel, terdapat beberapa nilai yang kosong karena field tersebut tidak dimiliki oleh semua dokumen. Selain itu, Spark membutuhkan tipe data yang konsisten untuk setiap kolom agar proses konversi dan analisis dapat berjalan dengan baik.

Cara mengatasinya adalah dengan mengubah dokumen MongoDB menjadi pandas DataFrame terlebih dahulu, kemudian memastikan seluruh kolom yang dibutuhkan tersedia. Field yang tidak dimiliki suatu dokumen dibiarkan bernilai null, bukan diisi dengan data yang tidak sesuai. Setelah itu, saya menentukan skema Spark secara eksplisit, seperti IntegerType untuk harga dan tingkat kepahitan serta StringType untuk nama menu, kategori, dan status. Dengan cara ini, data dari MongoDB dapat dikonversi ke Spark DataFrame dan dianalisis menggunakan groupBy meskipun struktur awal dokumennya berbeda.

In [15]:
# 1. Definisikan kembali koneksi dan variabel 'menu'
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
menu = db["menu"]

# 2. Jalankan kueri spesifik Anda
print("\n--- DAFTAR MENU TERMURAH ---")
for item in menu.find({}, {"_id": 0, "nama_menu": 1, "harga": 1}).sort("harga", 1):
    print(item)

print("\n--- MENU MAKANAN TIDAK PEDAS ---")
for item in menu.find({"kategori": "Makanan", "pedas_level": 0}, {"_id": 0, "nama_menu": 1}):
    print(item)

print("\n--- TOTAL STOK PER KATEGORI ---")
for hasil in menu.aggregate([{"$group": {"_id": "$kategori", "total": {"$sum": "$stok"}}}]):
    print(hasil)


--- DAFTAR MENU TERMURAH ---
{'nama_menu': 'Kopi Tubruk', 'harga': 15000}
{'nama_menu': 'Pisang Goreng', 'harga': 15000}
{'nama_menu': 'Espresso', 'harga': 18000}
{'nama_menu': 'Roti Bakar Cokelat', 'harga': 18000}
{'nama_menu': 'Singkong Keju', 'harga': 20000}
{'nama_menu': 'Americano', 'harga': 22000}
{'nama_menu': 'Mie Goreng Jawa', 'harga': 25000}
{'nama_menu': 'Cafe Latte', 'harga': 27000}
{'nama_menu': 'Cappuccino', 'harga': 28000}
{'nama_menu': 'Nasi Goreng Nusantara', 'harga': 28000}
{'nama_menu': 'Mocha', 'harga': 30000}
{'nama_menu': 'Chicken Rice Bowl', 'harga': 30000}

--- MENU MAKANAN TIDAK PEDAS ---
{'nama_menu': 'Roti Bakar Cokelat'}
{'nama_menu': 'Pisang Goreng'}
{'nama_menu': 'Singkong Keju'}

--- TOTAL STOK PER KATEGORI ---
{'_id': 'Makanan', 'total': 81}
{'_id': 'Kopi', 'total': 130}


In [16]:
# Mencari menu Kopi yang rasanya manis-sedang (Tingkat Kepahitan di bawah atau sama dengan 3)
print("\n--- KOPI DENGAN KEPREDIKSIAN MANIS-SEDANG ---")
for item in menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$lte": 3}}, {"_id": 0, "nama_menu": 1, "tingkat_kepahitan": 1}):
    print(item)

# Menghitung Rata-rata Harga dan Stok Tertinggi Menggunakan MongoDB Aggregation
print("\n--- STATISTIK BISNIS (MONGODB AGGREGATION) ---")
pipelining_baru = [
    {
        "$group": {
            "_id": "$kategori",
            "rata_rata_harga": {"$avg": "$harga"},
            "stok_paling_banyak": {"$max": "$stok"}
        }
    }
]
for hasil in menu.aggregate(pipelining_baru):
    print(hasil)


--- KOPI DENGAN KEPREDIKSIAN MANIS-SEDANG ---
{'nama_menu': 'Cappuccino', 'tingkat_kepahitan': 3}
{'nama_menu': 'Cafe Latte', 'tingkat_kepahitan': 2}
{'nama_menu': 'Mocha', 'tingkat_kepahitan': 3}

--- STATISTIK BISNIS (MONGODB AGGREGATION) ---
{'_id': 'Kopi', 'rata_rata_harga': 23333.333333333332, 'stok_paling_banyak': 30}
{'_id': 'Makanan', 'rata_rata_harga': 22666.666666666668, 'stok_paling_banyak': 20}


In [17]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
